# Project 1: DQN — Playing Flappy Bird from Pixels

**Course:** Reinforcement Learning, Summer 2026 — Kyiv School of Economics  
**Instructor:** Yurii Hannich

## Overview

In this project you will reproduce the core ideas from the foundational DQN paper:

> Mnih, V. et al. (2013). *Playing Atari with Deep Reinforcement Learning.* [arXiv:1312.5602](https://arxiv.org/abs/1312.5602)

You will train a Deep Q-Network to play **Flappy Bird from raw pixels** — the same preprocessing pipeline, architecture and training described in the paper, adapted to the FlappyBird-v0 environment.

### What you will implement:

| # | Task | What |
|---|------|------|
| 1 | `make_env()` | Environment with DQN preprocessing (Gymnasium wrappers) |
| 2 | `make_dqn()` | Convolutional neural network (2013 architecture) |
| 3 | `get_epsilon()` | Epsilon decay schedule |
| 4 | DQN training | Full training loop from paper algorithm and hyperparameters|
| 5 | Target network + `model.pt` | Add target network → trained agent that passes performance threshold |

### Grading

All-or-nothing: pass all automated tests → **15 pts**, fail any → **0 pts**.

Submit by pushing your completed notebook + `model.pt` to your GitHub repo.

## Setup

In [ ]:
!uv pip install torch numpy gymnasium flappy-bird-gymnasium matplotlib tqdm

In [ ]:
# Mount Google Drive to save models (optional)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"  # headless rendering (no display needed)

import gymnasium as gym
import flappy_bird_gymnasium  # registers FlappyBird-v0
import numpy as np
import matplotlib.pyplot as plt
import random
import torch

from tqdm.auto import tqdm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Reproducibility
seed = 1234
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Gymnasium version: {gym.__version__}")

## Section 1: Environment Exploration

[FlappyBird-v0](https://github.com/markub3327/flappy-bird-gymnasium) is a simple 2D game where the agent controls a bird flying through pipes.

**Actions:** `Discrete(2)` — 0 = do nothing, 1 = flap

**Rewards:**
- `+0.1` every step the bird stays alive
- `+1.0` for passing through a pipe
- `-1.0` for dying (hitting a pipe or the ground)
- `-0.5` per step when touching the ceiling

**Episode termination (`terminated=True`):** the bird hits a pipe or the ground.  
**Episode truncation (`truncated=True`):** the score reaches the maximum limit.

**Default observation:** A 12-element numeric vector (pipe positions, velocity, rotation).  
But the DQN paper uses **raw pixel observations** — so we need to convert the environment to output RGB frames instead.

Let's explore both modes.

In [ ]:
# Explore the default (numeric) FlappyBird environment
env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)

print("=" * 55)
print("FLAPPY BIRD — DEFAULT OBSERVATION")
print("=" * 55)
print(f"Observation space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"  Dtype: {env.observation_space.dtype}")
print()
print(f"Action space: {env.action_space}")
print(f"  n={env.action_space.n}  (0=idle, 1=flap)")
print()

obs, info = env.reset(seed=seed)
print(f"Sample observation (12 numbers): {obs}")
print()

# Show what the game looks like
frame = env.render()
print(f"Rendered frame shape: {frame.shape}  (H x W x C)")

plt.figure(figsize=(4, 6))
plt.imshow(frame)
plt.axis("off")
plt.title("FlappyBird-v0 — rendered frame (512x288 RGB)")
plt.tight_layout()
plt.show()

env.close()

### The Problem: DQN Needs Pixel Observations

The DQN paper trains directly from **raw pixels** — the agent sees the screen image, not hand-crafted features. It's a big advantage as in most real-world cases when we need AI agent to operate in the world we will have just some raw sensory data.

FlappyBird's default observation is a 12-number vector (pipe positions, bird velocity, etc.). We need to replace this with the rendered RGB frame.

We solve this with a custom **`ObservationWrapper`** — a pattern you already know from Workshop 1:

In [ ]:
class RenderAsObsWrapper(gym.ObservationWrapper):
    """Replace the numeric observation with the rendered RGB frame.

    FlappyBird-v0 default obs is a 12-element vector.
    DQN needs pixel input (H x W x C array). This wrapper bridges the gap
    by returning env.render() as the observation.

    The environment MUST be created with render_mode='rgb_array'.
    """
    def __init__(self, env):
        super().__init__(env)
        # Determine frame shape by rendering once
        env.reset()
        frame = env.render()
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=frame.shape, dtype=np.uint8
        )

    def observation(self, obs):
        return self.env.render()


print("RenderAsObsWrapper — defined ✓")

In [ ]:
# Verify: now the observation IS the pixel frame
env_pixels = RenderAsObsWrapper(
    gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)
)

print(f"Wrapped observation space: {env_pixels.observation_space.shape}")
print(f"Dtype: {env_pixels.observation_space.dtype}")
print()

obs, _ = env_pixels.reset(seed=seed)
print(f"Observation shape: {obs.shape}")
print(f"Observation dtype: {obs.dtype}")
print(f"Value range: [{obs.min()}, {obs.max()}]")

plt.figure(figsize=(4, 6))
plt.imshow(obs)
plt.axis("off")
plt.title(f"Observation IS the frame now — shape {obs.shape}")
plt.tight_layout()
plt.show()

env_pixels.close()

## Section 2: Preprocessing Pipeline (Task 1)

The DQN paper describes how raw Atari frames are preprocessed before being fed to the network.

Your task: implement `make_env()` that applies the same preprocessing logic to FlappyBird, using Gymnasium observation wrappers.

**Notes:**
- The paper's cropping step (to make frames square) is unnecessary here, so you can resize directly to the target resolution.
- `RenderAsObsWrapper` is already provided above — build your pipeline on top of it.
- Think about what the network needs: what should the final observation shape be, and why?

In [ ]:
def make_env():
    """Create a FlappyBird environment with DQN preprocessing.

    Returns:
        gym.Env with DQN preprocessing.
    """
    env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)
    env = RenderAsObsWrapper(env)                        # numeric obs → RGB pixels (512, 288, 3)
    # --- Fill in below ---
    pass
    # --- Fill in above ---
    return env
    

print("make_env() — defined ✓")

In [ ]:
# Verify the preprocessing pipeline
env = make_env()

obs, _ = env.reset(seed=seed)
print(f"Observation shape: {obs.shape}")   # Expected: (4, 84, 84)
print(f"Observation dtype: {obs.dtype}")   # Expected: uint8
print(f"Value range: [{obs.min()}, {obs.max()}]")
print()

# Take a step to verify consistency
obs2, reward, terminated, truncated, info = env.step(0)
print(f"After step — shape: {obs2.shape}, reward: {reward}")
print()

# Visualize the 4 stacked frames
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(obs[i], cmap="gray")
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.suptitle("Stacked observation (4 × 84 × 84 grayscale frames)")
plt.tight_layout()
plt.show()

env.close()
print("\n✓ Pipeline working — ready for DQN!")

### Frame Skipping

Notice how the 4 stacked frames above are nearly identical — the bird barely moves between consecutive steps. The agent gets almost no temporal information from the stack.

The original DQN paper uses **frame skipping (k=4)** — repeating the agent's chosen action for k consecutive frames. This makes training faster (fewer decisions per episode) and gives the agent more temporal context.

We adopt frame skipping here too, but with a FlappyBird-specific twist: instead of repeating the agent's action, we execute it **once** and then idle (action=0) for the remaining k-1 frames. Why?

1. **Temporal diversity:** With skip=4, each frame in the stack is 4 physics steps apart, giving the agent real motion information instead of 4 almost identical images.
2. **Game pacing:** From the start of flight to the first pipe encounter is exactly 50 steps — that's a lot of "nothing happening" decisions. Frame skipping makes the effective game speed closer to what a human perceives.
3. **Easier early exploration:** During random exploration ($\varepsilon \approx 1$), the agent flaps excessively, reaching the top of screen. With frame skipping, each flap is followed by 3 idle frames — gravity pulls the bird back down naturally, making random trajectories longer and more informative.
4. **Training speed:** Skipped frames are never rendered — only 1 render per 4 physics steps → ~4× fewer expensive render calls.

Here's the wrapper (provided for you). It needs to sit **before** `RenderAsObsWrapper` so skipped frames are never rendered:

In [ ]:
class FrameSkipWrapper(gym.Wrapper):
    """Skip frames: execute agent's action, then idle (action=0) for (skip-1) frames.

    Placed BELOW RenderAsObsWrapper so skipped frames are never rendered.
    Rewards are accumulated across all k sub-steps.
    """

    def __init__(self, env, skip=4):
        super().__init__(env)
        self.skip = skip

    def step(self, action):
        total_reward = 0.0
        for i in range(self.skip):
            obs, reward, terminated, truncated, info = self.env.step(action if i == 0 else 0)
            total_reward += reward
            if terminated or truncated:
                break
        return obs, total_reward, terminated, truncated, info


print(f"FrameSkipWrapper — defined ✓ (skip=4)")

Now redefine `make_env()` to include `FrameSkipWrapper` as the first wrapper in the stack:

In [ ]:
def make_env():
    """Create a FlappyBird environment with DQN preprocessing + frame skipping.

    Returns:
        gym.Env with DQN preprocessing and frame skipping.
    """
    env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)
    # --- Fill in below ---
    pass
    # --- Fill in above ---
    return env


# Test the updated pipeline
env = make_env()
obs, _ = env.reset(seed=seed)
print(f"Observation shape: {obs.shape}")  # (4, 84, 84)

# Take a few steps to fill the stack with diverse frames
for _ in range(3):
    obs, _, _, _, _ = env.step(0)

# Visualize — frames should now show visible differences
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(obs[i], cmap="gray")
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.suptitle("With frame skip=4: stacked frames (each 4 physics steps apart)")
plt.tight_layout()
plt.show()

env.close()
print("\n✓ make_env() redefined with FrameSkipWrapper — ready for DQN!")
print("\n✓ If you start seeing the pipe on the right, everything is working correctly!")

## Section 3: DQN Network (Task 2)

In tabular Q-learning, we stored Q-values in a table — one entry per (state, action) pair. That's feasible for small discrete state spaces, but completely impossible when observations are raw pixel images (the state space is astronomically large).

The key insight of DQN: use a neural network as a **function approximator** for Q-values. Instead of a lookup table, we train a network that takes a state and outputs Q-values for all actions in a single forward pass. As Deep Neural Networks are quite good in preprocessing high dimensional data by extracting useful patterns and discarding noise, we can use a neural network to map raw pixels into some meaningful features space.

The paper uses a convolutional architecture — convolutions extract spatial features from pixel inputs, then fully-connected layers map those features to Q-values.

Implement the exact DQN architecture from the paper as a `torch.nn.Module`. It's a straightforward sequential CNN — nothing exotic.

In [ ]:
import torch.nn as nn


def make_dqn(n_actions):
    """Create the DQN network (Mnih et al., 2013 architecture).

    Args:
        n_actions: Number of possible actions.

    Returns:
        nn.Sequential network: input (batch, 4, 84, 84) → output (batch, n_actions)
    """
    # --- Fill in below ---
    pass
    # --- Fill in above ---


print("make_dqn() — defined ✓")

In [ ]:
# Verify DQN architecture
dqn = make_dqn(n_actions=2).to(device)

# Test forward pass
dummy_input = torch.randint(0, 255, (1, 4, 84, 84), dtype=torch.float32).to(device)
output = dqn(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")  # Expected: (1, 2)
print(f"Output values (Q-values): {output.detach().cpu().numpy()}")
print()

total_params = sum(p.numel() for p in dqn.parameters())
print(f"Total parameters: {total_params:,}")
assert total_params == 676_658, f"Expected 676,658 params, got {total_params:,}"
print("✓ Parameter count matches DQN 2013 architecture")
print()
print(dqn)

## Section 4: Experience Replay

Training a neural network on consecutive gameplay transitions is problematic — successive frames are highly correlated, and the data distribution shifts as the policy changes. This violates the i.i.d. assumption that SGD relies on.

The DQN paper solves this with **experience replay**: store transitions in a buffer, then train on random mini-batches sampled from it. This breaks temporal correlations and reuses each experience multiple times.

Implement a `ReplayBuffer` class with the following interface:
- `__init__(self, capacity)` — maximum number of transitions to store
- `push(self, state, action, reward, next_state, terminated, truncated)` — add a transition (FIFO: drop oldest when full)
- `sample(self, batch_size)` — return a random batch of transitions
- `__len__(self)` — current number of stored transitions

In [ ]:
class ReplayBuffer:
    """Fixed-size replay buffer using pre-allocated tensors on device (GPU/CPU).

    Stores transitions in circular tensors — no Python object overhead,
    no memory fragmentation, no hidden numpy views.
    """

    def __init__(self, capacity, obs_shape=(4, 84, 84), device=device):
        self.capacity = capacity
        self.device = device
        self.pos = 0
        self.size = 0

        # Pre-allocate all storage on device (uint8 for frames to save memory)
        self.states = torch.zeros((capacity, *obs_shape), dtype=torch.uint8, device=device)
        self.next_states = torch.zeros((capacity, *obs_shape), dtype=torch.uint8, device=device)
        self.actions = torch.zeros(capacity, dtype=torch.long, device=device)
        self.rewards = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.terminateds = torch.zeros(capacity, dtype=torch.bool, device=device)

    def push(self, state, action, reward, next_state, terminated, truncated):
        self.states[self.pos] = torch.as_tensor(np.asarray(state), device=self.device)
        self.next_states[self.pos] = torch.as_tensor(np.asarray(next_state), device=self.device)
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.terminateds[self.pos] = terminated
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = torch.randint(0, self.size, (batch_size,), device=self.device)
        return (
            self.states[indices].float(),
            self.actions[indices],
            self.rewards[indices],
            self.next_states[indices].float(),
            self.terminateds[indices].float(),
        )

    def __len__(self):
        return self.size


print(f"ReplayBuffer — defined ✓ (device: {device})")


In [ ]:
# Verify ReplayBuffer
buf = ReplayBuffer(capacity=100, device=device)

# Push some dummy transitions
dummy_state = np.zeros((4, 84, 84), dtype=np.uint8)
for i in range(50):
    buf.push(dummy_state + i, i % 2, 0.1 * i, dummy_state + i + 1, i == 49, False)

print(f"Buffer size: {len(buf)}")  # Expected: 50

# Sample a batch
states, actions, rewards, next_states, terminateds = buf.sample(8)
print(f"Batch shapes:")
print(f"  states:      {states.shape}")       # (8, 4, 84, 84)
print(f"  actions:     {actions.shape}")      # (8,)
print(f"  rewards:     {rewards.shape}")      # (8,)
print(f"  next_states: {next_states.shape}")  # (8, 4, 84, 84)
print(f"  terminateds: {terminateds.shape}")  # (8,)
print(f"  Device:      {states.device}")
print()

# Test FIFO overflow
buf_small = ReplayBuffer(capacity=5, device=device)
for i in range(10):
    buf_small.push(np.zeros((4,84,84), dtype=np.uint8), 0, 0.0, np.zeros((4,84,84), dtype=np.uint8), False, False)
print(f"Overflow test — capacity 5, pushed 10: len={len(buf_small)}")
assert len(buf_small) == 5
print("✓ ReplayBuffer working correctly")


## Section 5: Epsilon-Greedy Exploration (Task 3)

DQN uses an **epsilon-greedy** policy during training: with probability $\varepsilon$ take a random action (explore), otherwise take the greedy action $\arg\max_a Q(s, a)$ (exploit).

Early in training the network knows nothing — random exploration is essential. As training progresses and Q-values become more accurate, we reduce $\varepsilon$ to exploit learned knowledge more often.

The paper uses some specific $\varepsilon$ scheduling strategy.

Implement `get_epsilon(step, eps_start, eps_end, eps_decay_steps)` — returns the current epsilon value for a given training step.

In [ ]:
def get_epsilon(step, eps_start, eps_end, eps_decay_steps):
    """Epsilon decay schedule.

    Args:
        step: Current training step.
        eps_start: Starting epsilon (step 0).
        eps_end: Minimum epsilon (after decay_steps).
        eps_decay_steps: Number of steps over which to decay.

    Returns:
        float: Current epsilon value.
    """
    # --- Fill in below ---
    pass
    # --- Fill in above ---


print("get_epsilon() — defined ✓")

In [ ]:
# Verify epsilon schedule
eps_start, eps_end, eps_decay_steps = 1.0, 0.01, 100_000

assert get_epsilon(0, eps_start, eps_end, eps_decay_steps) == eps_start
assert abs(get_epsilon(eps_decay_steps, eps_start, eps_end, eps_decay_steps) - eps_end) < 1e-9
assert abs(get_epsilon(eps_decay_steps * 2, eps_start, eps_end, eps_decay_steps) - eps_end) < 1e-9

# Epsilon should decrease monotonically
prev = get_epsilon(0, eps_start, eps_end, eps_decay_steps)
for s in range(1000, eps_decay_steps, 1000):
    curr = get_epsilon(s, eps_start, eps_end, eps_decay_steps)
    assert curr <= prev, f"Epsilon should not increase: step {s}"
    prev = curr

print("✓ All epsilon checks passed")
print()

# Visualize the schedule
steps = range(0, eps_decay_steps + 50_000, 100)
epsilons = [get_epsilon(s, eps_start, eps_end, eps_decay_steps) for s in steps]

plt.figure(figsize=(8, 3))
plt.plot(steps, epsilons)
plt.xlabel("Training step")
plt.ylabel("Epsilon (ε)")
plt.title("Epsilon decay schedule")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 6: Training (Task 4)

Now you have all the building blocks — environment, network, replay buffer, and epsilon schedule. Time to put them together into the full DQN training loop from Algorithm 1 in the paper.

Below are the training hyperparameters. Some are specified in the paper — you must find and set them correctly. Two constants (learning rate and discount factor) are **not stated** in the paper and are provided for you.

**Note:** The authors of paper have trained their DQN using quite big replay buffer and number of training steps, which may be challenging for us. So we significantly down-scale all the replay-related hyperparameters to make training feasible on a single T4 GPU in short time.

**Important — paper technique we do NOT use** here is **Reward clipping**. The paper clips all rewards to [-1, +1] because it trains a single network across many different Atari games with different score scales. FlappyBird rewards are already small and meaningful (+0.1 alive, +1.0 pipe, -1.0 death, -0.5 for touching the ceiling - just to push initial random policy to not flap to much), so clipping would destroy useful signal.

> ⏱ **Training time:** expect ~30 minutes for full training on a free Colab T4 GPU.

### Hyperparameters setting

In [ ]:
# === Training Hyperparameters ===

# --- Fill in below ---
# From the paper — find the correct values:
BATCH_SIZE = ...                     
EPS_START = ...                     
EPS_END = ...

# set x100 smaller than in the paper — find the correct values, then down-scale
TRAINING_STEPS = ...
# set x50 smaller than in the paper — find the correct values, then down-scale
REPLAY_MEMORY_SIZE = ...
EPS_DECAY_STEPS = ...

# NOT specified in the paper — provided for you:
LEARNING_RATE = 0.0001             
GAMMA = 1.0                         

# Initialize DQN, optimizer, and loss function
torch.manual_seed(seed)
dqn = make_dqn(n_actions=2).to(device)

# Select the correct optimizer and loss function from the paper
optimizer = ...
loss_fn = ...
# --- Fill in above ---

print("Training setup:")
print(f"  Batch size:       {BATCH_SIZE}")
print(f"  ε:                {EPS_START} → {EPS_END} over {EPS_DECAY_STEPS:,} steps")
print(f"  Training steps:   {TRAINING_STEPS:,}")
print(f"  Replay memory:    {REPLAY_MEMORY_SIZE:,}")
print(f"  Learning rate:    {LEARNING_RATE}")
print(f"  Gamma (γ):        {GAMMA}")
print(f"  Device:           {device}")

### Training

In [ ]:
# === DQN Training Loop ===
# Implements Deep Q-Learning algorithm from the paper.
torch.manual_seed(seed)

env = make_env()
replay_buffer = ReplayBuffer(REPLAY_MEMORY_SIZE, device=device)

# Tracking
episode_rewards = []
current_episode_reward = 0.0
episode_count = 0

obs, _ = env.reset(seed=seed)

# --- Fill in below ---
pass
# --- Fill in above ---

env.close()
print(f"\n✓ Training complete — {episode_count} episodes, {TRAINING_STEPS:,} steps")


In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.3, label="Episode reward")

# Smoothed curve (moving average)
if len(episode_rewards) >= 20:
    smoothed = np.convolve(episode_rewards, np.ones(20)/20, mode='valid')
    plt.plot(range(19, len(episode_rewards)), smoothed, linewidth=2, label="20-episode running average")

plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN Training Progress — FlappyBird")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final 20-episode running average: {np.mean(episode_rewards[-20:]):.2f}")

### Evaluation

Run the trained agent with **deterministic actions** (no exploration, always pick $\arg\max_a Q(s, a)$) to measure its true performance.

We evaluate over **100 episodes** with different random seeds to get a reliable estimate — mean reward, standard deviation, and the full distribution.

In [ ]:
# Evaluate the trained DQN — 100 episodes with different seeds
N_EVAL_EPISODES = 100
eval_rewards = []

for ep in tqdm(range(N_EVAL_EPISODES), desc="Evaluating"):
    eval_env = make_env()
    obs, _ = eval_env.reset(seed=ep)
    episode_reward = 0.0
    done = False

    while not done:
        with torch.no_grad():
            state_t = torch.tensor(np.array(obs), dtype=torch.float32).unsqueeze(0).to(device)
            action = dqn(state_t).argmax(dim=1).item()

        obs, reward, terminated, truncated, info = eval_env.step(action)
        episode_reward += reward
        done = terminated or truncated

    eval_env.close()
    eval_rewards.append(episode_reward)

eval_rewards = np.array(eval_rewards)
print(f"Evaluation (deterministic policy, {N_EVAL_EPISODES} episodes):")
print(f"  Mean reward:  {eval_rewards.mean():.2f} ± {eval_rewards.std():.2f}")
print(f"  Min / Max:    {eval_rewards.min():.2f} / {eval_rewards.max():.2f}")
print(f"  Median:       {np.median(eval_rewards):.2f}")

In [ ]:
# Distribution of evaluation rewards
plt.figure(figsize=(10, 4))
plt.hist(eval_rewards, bins=20, edgecolor="black", alpha=0.7)
plt.axvline(eval_rewards.mean(), color="red", linestyle="--", label=f"Mean: {eval_rewards.mean():.2f}")
plt.xlabel("Episode Reward")
plt.ylabel("Count")
plt.title("DQN (no target net) — Reward Distribution over 100 Episodes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Render a single episode as video (qualitative evaluation)
eval_env = make_env()
obs, _ = eval_env.reset(seed=0)

frames = []
episode_reward = 0.0
done = False

while not done:
    frame = eval_env.render()
    frames.append(frame)

    with torch.no_grad():
        state_t = torch.tensor(np.array(obs), dtype=torch.float32).unsqueeze(0).to(device)
        action = dqn(state_t).argmax(dim=1).item()

    obs, reward, terminated, truncated, info = eval_env.step(action)
    episode_reward += reward
    done = terminated or truncated

eval_env.close()
print(f"Episode reward: {episode_reward:.2f}, length: {len(frames)} frames")

# Render as video
fig, ax = plt.subplots(figsize=(4, 6))
ax.axis("off")
img = ax.imshow(frames[0])

def update(i):
    img.set_data(frames[i])
    return [img]

anim = FuncAnimation(fig, update, frames=len(frames), interval=45, blit=True)
plt.close()
HTML(anim.to_html5_video())

### Expected Results

> **Expected learning curve**: The training curve is noisy and oscillates — a symptom of the moving-target problem where the same network is used for both prediction and TD target
>
> **Expected reward**: median reward `~40` (depends on initialisation but must be somewhere around)
>
> **Expected agent behaviour**: The agent has learned a quite good policy and is capable to pass the pipes and survive for a long time

## Section 7: Stabilising Training with a Target Network (Task 5)

You may notice that training in Section 6 is unstable — reward oscillates and Q-values can diverge. This happens because the TD target $y = r + \gamma \max_{a'} Q(s', a'; \theta)$ uses the **same network** we are updating. Every gradient step shifts both the prediction and the target, creating a moving-target problem.

In 2015, the same authors addressed this in:

> Mnih, V. et al. (2015). *Human-level control through deep reinforcement learning.* [Nature 518, 529–533](https://web.stanford.edu/class/psych209/Readings/MnihEtAlHassibis15NatureControlDeepRL.pdf)

They introduced a **target network** $Q(s, a; \theta^-)$ — a frozen copy of the Q-network that is only updated every $C$ steps. The TD target becomes:

$$y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$$

This decouples the target from the current parameters, stabilising learning significantly.

**Your task:** modify your training loop to use a target network. Use `TARGET_UPDATE_FREQ` below as the update interval $C$. Everything else (hyperparameters, environment, replay buffer) stays the same.

### Additional Hyperparameters

In [ ]:
TARGET_UPDATE_FREQ = 1000  # Copy θ → θ⁻ every C steps

# Initialize networks
dqn = make_dqn(n_actions=2).to(device)
target_dqn = make_dqn(n_actions=2).to(device)
target_dqn.load_state_dict(dqn.state_dict())

# --- Fill in below ---
optimizer = ...
loss_fn = ...
# --- Fill in above ---

### Training

In [ ]:
# === Training with Target Network ===
torch.manual_seed(seed)

env = make_env()
replay_buffer = ReplayBuffer(REPLAY_MEMORY_SIZE, device=device)

# Tracking
episode_rewards_target = []
current_episode_reward = 0.0
episode_count = 0

obs, _ = env.reset(seed=seed)

# --- Fill in below ---
pass
# --- Fill in above ---

env.close()
print(f"\n✓ Training complete — {episode_count} episodes, {TRAINING_STEPS:,} steps")


In [ ]:
# Visualize training progress (target network)
plt.figure(figsize=(10, 4))
plt.plot(episode_rewards_target, alpha=0.3, label="Episode reward")

if len(episode_rewards_target) >= 20:
    smoothed = np.convolve(episode_rewards_target, np.ones(20)/20, mode='valid')
    plt.plot(range(19, len(episode_rewards_target)), smoothed, linewidth=2, label="20-episode rolling average")

plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN + Target Network — Training Progress")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final 20-episode rolling average: {np.mean(episode_rewards_target[-20:]):.2f}")

### Evaluation (Target Network DQN)

Same 100-episode evaluation for the target-network DQN:

In [ ]:
# Evaluate the DQN with target network — 100 episodes with different seeds
N_EVAL_EPISODES = 100
eval_rewards_tn = []

for ep in tqdm(range(N_EVAL_EPISODES), desc="Evaluating (Target Net)"):
    eval_env = make_env()
    obs, _ = eval_env.reset(seed=ep)
    episode_reward = 0.0
    done = False

    while not done:
        with torch.no_grad():
            state_t = torch.tensor(np.array(obs), dtype=torch.float32).unsqueeze(0).to(device)
            action = dqn(state_t).argmax(dim=1).item()

        obs, reward, terminated, truncated, info = eval_env.step(action)
        episode_reward += reward
        done = terminated or truncated

    eval_env.close()
    eval_rewards_tn.append(episode_reward)

eval_rewards_tn = np.array(eval_rewards_tn)
print(f"Evaluation (deterministic policy, {N_EVAL_EPISODES} episodes):")
print(f"  Mean reward:  {eval_rewards_tn.mean():.2f} ± {eval_rewards_tn.std():.2f}")
print(f"  Min / Max:    {eval_rewards_tn.min():.2f} / {eval_rewards_tn.max():.2f}")
print(f"  Median:       {np.median(eval_rewards_tn):.2f}")

In [ ]:
# Distribution of evaluation rewards (Target Network)
plt.figure(figsize=(10, 4))
plt.hist(eval_rewards_tn, bins=20, edgecolor="black", alpha=0.7)
plt.axvline(eval_rewards_tn.mean(), color="red", linestyle="--", label=f"Mean: {eval_rewards_tn.mean():.2f}")
plt.xlabel("Episode Reward")
plt.ylabel("Count")
plt.title("DQN + Target Network — Reward Distribution over 100 Episodes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Render a single episode as video (qualitative evaluation)
eval_env = make_env()
obs, _ = eval_env.reset(seed=0)

frames = []
episode_reward = 0.0
done = False

while not done:
    frame = eval_env.render()
    frames.append(frame)

    with torch.no_grad():
        state_t = torch.tensor(np.array(obs), dtype=torch.float32).unsqueeze(0).to(device)
        action = dqn(state_t).argmax(dim=1).item()

    obs, reward, terminated, truncated, info = eval_env.step(action)
    episode_reward += reward
    done = terminated or truncated

eval_env.close()
print(f"Episode reward: {episode_reward:.2f}, length: {len(frames)} frames")

# Render as video
fig, ax = plt.subplots(figsize=(4, 6))
ax.axis("off")
img = ax.imshow(frames[0])

def update(i):
    img.set_data(frames[i])
    return [img]

anim = FuncAnimation(fig, update, frames=len(frames), interval=45, blit=True)
plt.close()
HTML(anim.to_html5_video())

### Expected Results

> **Expected learning curve**: Learning is still noisy but a bit more stable and reward is oscillating less
>
> **Expected reward**: median reward `~50` (depends on initialisation but must be somewhere around)
>
> **Expected agent behaviour**: Basically similar behaviour that we observe in previous training

The target network decouples the TD target from the parameters being updated, removing the moving-target instability. Same hyperparameters, same environment — the only difference is the frozen target network updated every $C=1000$ steps

The reason why we still have so unstable training is lack of normalization of observations and target, no control over gradients and learning rate, simple version of algorithm.

## Section 8: Model Saving

In [ ]:
# Save the target-network trained model (this is the final submission)
torch.save(dqn.state_dict(), "model.pt")
print(f"✓ Model saved to model.pt")
print(f"  File size: {os.path.getsize('model.pt') / 1024:.1f} KB")

In [ ]:
# Save to Google Drive (optional)
!cp model.pt /content/drive/MyDrive/model.pt
print("✓ Model copied to Google Drive → MyDrive/model.pt")

In [ ]:
# Unassign the Colab runtime to free resources (optional)
from google.colab import runtime
runtime.unassign()

## Why can be improved?

Even with 100k steps of training, both agents could learn how to play the game. But we can assume that there are a lot of human players who can play much better. So why we haven't achieved Superhuman Level Performance?

The original DQN paper trained on **much more frames** with a **much bigger replay buffer**. We deliberately scaled everything down to keep training around 30 minutes on a free Colab GPU. At this scale the agent sees roughly the same amount of data as an Atari agent sees in its first 1% of training.

**What would help:**
- **More training steps** — the single most impactful change. 500k–1M steps would let the agent pass more pipes consistently
- **Larger replay buffer** — more diverse experience to sample from reduces overfitting to recent transitions
- **Longer epsilon decay** — slower decay gives the agent more exploration time before committing to a policy
- **Bigger network** — our model is tiny with a simple architecture; a bigger one with more advanced architecture could extract richer features from the frames
- **Hyperparameter tuning** — learning rate scheduling, gradient clipping, or switching to another optimizer can improve stability
- **Reward shaping** — engineering more informative reward signals so the agent gets clearer feedback on what's good before reaching a pipe
- **Observation normalisation** — scaling pixel values to [0, 1] or standardising them helps gradient flow and speeds up convergence
- **Reward normalisation / clipping** — keeping reward magnitudes consistent prevents large TD targets from destabilising updates
- **More advanced Q-learning variants** — Double DQN (reduces overestimation bias), Dueling DQN (separates state-value from action-advantage), or Prioritised Experience Replay (samples important transitions more often)

The point of this project is not to achieve superhuman performance — it's to implement the algorithm correctly and observe the **relative difference** between vanilla DQN (unstable) and DQN + target network (stable).

We will apply some of improvements listed above in **Workshop 5** using the **Stable-Baselines3** library — which bundles well-tested implementations of DQN, PPO, and other algorithms out of the box.

## What You've Learned

In this project you went from a raw research paper to a working RL agent. That means you practiced three different skill sets:

**Theory** — deep Q-learning fundamentals: function approximation for Q-values, experience replay, epsilon-greedy exploration, the moving-target problem, and how target networks solve it.

**Engineering** — building a full pixel-to-action training pipeline: frame preprocessing with Gymnasium wrappers, CNN architecture in PyTorch, replay buffer, training loop and evaluation.

**Research** — reading a foundational paper (Mnih et al., 2013), extracting the algorithm, translating pseudocode into code, then comparing your results against the follow-up paper (2015 Nature) to see how target networks stabilise learning.